### We will construct a linear model that can predict a insurance Premium amount by using its other attributes.

### Data Descrpition

The dataset has 7 variables.

1. age: age of the policyholder. 
2. 
sex: gender of the policyholde. r
3. 
bmi: Body Mass Index of the policyhold. e
4. r
children: number of children of the policyhol. d
5. er
smoker: whether the policyholder is a smoker or.  
6. not
region: region where the policyholder belon. g
7. s to
charges: premium charged to the policy. holder


### Import Libraries

In [ ]:
import pandas as pd
import numpy as np

# for visualizing data
import matplotlib.pyplot as plt
import seaborn as sns

# For randomized data splitting
from sklearn.model_selection import train_test_split

# To build linear regression_model
import statsmodels.api as sm

# To check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Load and explore the data

In [ ]:
cData = pd.read_csv("insurance.csv")

In [ ]:
cData.shape

In [ ]:
cData.head()

In [ ]:
cData.info()

* Most of the columns in the dataset are numeric in nature ('int64 or Float64')
* All are in correct datatypes

In [ ]:
# let's check the statistical summary of the data
cData.describe()

## Bivariate Analysis

A bivariate analysis among the different variables can be done using scatter matrix plot. Seaborn libs create a dashboard reflecting useful information about the dimensions. The result can be stored as a .png file.

In [ ]:
cData_attr = cData.iloc[:, 0:7]
sns.pairplot(
    cData_attr, diag_kind="kde"
) 

* By oberserving relationship between charge and other attributes are not really linear.
* serveral assumptions of classical linear regression seem to be vilotaed.

In [ ]:
# drop_first=True will drop one of the three origin columns
cData = pd.get_dummies(cData, columns=["sex"], drop_first=True)
cData.head()

In [ ]:
cData = pd.get_dummies(cData, columns=["smoker"], drop_first=True)
cData.head()

In [ ]:
cData = pd.get_dummies(cData, columns=["region"], drop_first=True)
cData.head()

In [ ]:
cData[["sex_male","smoker_yes","region_northwest","region_southeast","region_southwest"]] = cData[["sex_male","smoker_yes","region_northwest","region_southeast","region_southwest"]].replace({True: 1, False: 0})
cData.head()

### Split Data

In [ ]:
# independent variables
X = cData.drop(["charges"], axis=1)
# dependent variable
y = cData[["charges"]]

In [ ]:
# let's add the intercept to data
X = sm.add_constant(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1
)

In [ ]:
print(X_train.head())

In [ ]:
print(X_test.head())

## Fit Linear Model

In [ ]:
olsmod = sm.OLS(y_train, X_train)
olsres = olsmod.fit()

In [ ]:
# let's print the regression summary
print(olsres.summary())

## Checking for Multicollinearity

In [ ]:
# let's check the VIF of the predictors
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_series1 = pd.Series(
    [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])],
    index=X_train.columns,
)
print("VIF values: \n\n{}\n".format(vif_series1))

### Now that we do not have multicollinearity in our data, the p-values of the coefficients have become reliable and we can remove the non-significant predictor variables.

As observed in the above model (olsres), 'sex_male' has a p-value greater than 0.05. So, we can drop it because it is not significant in predicting 'charges'.

### TEST FOR LINEARITY AND INDEPENDENCE

In [ ]:
df_pred = pd.DataFrame()

df_pred["Actual Values"] = y_train.values.flatten()  # actual values
df_pred["Fitted Values"] = olsres.fittedvalues.values  # predicted values
df_pred["Residuals"] = olsres.resid.values  # residuals

df_pred.head()

In [ ]:
# let us plot the fitted values vs residuals
sns.set_style("whitegrid")
sns.residplot(
    data=df_pred, x="Fitted Values", y="Residuals", color="purple", lowess=True
)
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Fitted vs Residual plot")
plt.show()

In [ ]:
# columns in training set
X_train.columns

In [ ]:
# checking the distribution of variables in training set with dependent variable
sns.pairplot(cData[['charges','age', 'bmi', 'children', 'sex_male', 'smoker_yes',
       'region_northwest', 'region_southeast', 'region_southwest']])
plt.show()

In [ ]:
sns.histplot(df_pred["Residuals"], kde=True)
plt.title("Normality of residuals")
plt.show()

In [ ]:
import pylab
import scipy.stats as stats

stats.probplot(df_pred["Residuals"], dist="norm", plot=pylab)
plt.show()

In [ ]:
import statsmodels.stats.api as sms
from statsmodels.compat import lzip

In [ ]:
name = ["F statistic", "p-value"]
test = sms.het_goldfeldquandt(df_pred["Residuals"], X_train)
lzip(name, test)

## Predictions

In [ ]:
# let's check the model parameters
olsres.params

In [ ]:
X_train.columns

In [ ]:
X_test.columns

In [ ]:
# let's make predictions on the test set
y_pred = olsres.predict(X_test)

In [ ]:
# let's check the RMSE on the train data
rmse1 = np.sqrt(mean_squared_error(y_train, df_pred["Fitted Values"]))
rmse1

In [ ]:
# let's check the RMSE on the test data
rmse2 = np.sqrt(mean_squared_error(y_test, y_pred))
rmse2

In [ ]:
# let's check the MAE on the train data
mae1 = mean_absolute_error(y_train, df_pred["Fitted Values"])
mae1

In [ ]:
# let's check the MAE on the test data
mae2 = mean_absolute_error(y_test, y_pred)
mae2